In [ ]:
!pip install ultralytics
!pip install torch torchvision opencv-python matplotlib segment-anything

In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth -O sam_vit_l.pth

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="You are using `torch.load` with `weights_only=False`")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import torch
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

# ==============================
#       GLOBAL SETTINGS
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

ROAD_COLOR = [128, 64, 128]      # Purple
GREEN_AREA_COLOR = [107, 142, 35]    # Green
BUILDING_COLOR = [70, 70, 70]      # Gray
UNKNOWN_COLOR = [0, 0, 0]   # Black

# ==============================
#       IMAGE UTILS
# ==============================
def resize_image(image, target_size=(1024, 1024)):
    resized_image = image.resize(target_size, Image.LANCZOS)
    return np.array(resized_image)

def overlay_masks_on_image(image, masks):
    """ Creates an image overlay of the segmentation masks on top of the original image. """
    overlay = image.copy()
    color_map = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]  # Different colors for different masks
    for i, mask_data in enumerate(masks):
        seg = mask_data["segmentation"]
        color = color_map[i % len(color_map)]  # Cycle through colors
        overlay[seg > 0] = color  # Apply color where segmentation is true
    return overlay

def display_images(original_image, resized_image, segmented_overlay, final_classified_image):
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    axes[0].imshow(original_image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(resized_image)
    axes[1].set_title("Resized Image (1024x1024)")
    axes[1].axis("off")

    axes[2].imshow(segmented_overlay)
    axes[2].set_title("SAM Segmentation Overlay")
    axes[2].axis("off")

    axes[3].imshow(final_classified_image.astype(np.uint8))
    axes[3].set_title("Final Classified Image")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

# ==============================
#       SAM MODEL UTILS
# ==============================
def load_sam_model(checkpoint_path):
    sam_model = sam_model_registry["vit_l"]()
    state_dict = torch.load(checkpoint_path, map_location=device)
    sam_model.load_state_dict(state_dict)
    sam_model = sam_model.to(device)
    mask_generator = SamAutomaticMaskGenerator(
        sam_model,
        points_per_side=64,
        pred_iou_thresh=0.90,
        stability_score_thresh=0.7
    )
    return mask_generator

def generate_masks(image, mask_generator):
    masks = mask_generator.generate(image)
    return masks

# ==============================
#       MASK MERGING UTILS
# ==============================
def compute_intersection(mask_a, mask_b):
    return np.logical_and(mask_a, mask_b).sum()

def merge_overlapping_masks(masks, overlap_threshold=0.5):
    segs = [mask['segmentation'].astype(bool) for mask in masks]
    n = len(segs)
    parent = list(range(n))

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for i in range(n):
        for j in range(i+1, n):
            area_i = segs[i].sum()
            area_j = segs[j].sum()
            inter = compute_intersection(segs[i], segs[j])
            if inter > 0:
                smaller_area = min(area_i, area_j)
                if inter / smaller_area > overlap_threshold:
                    union(i, j)

    clusters = {}
    for i in range(n):
        root = find(i)
        clusters.setdefault(root, []).append(i)

    merged_masks = []
    for indices in clusters.values():
        combined_seg = np.zeros_like(segs[0], dtype=bool)
        for idx in indices:
            combined_seg = np.logical_or(combined_seg, segs[idx])
        merged_masks.append({"segmentation": combined_seg.astype(np.uint8)})
    return merged_masks

# ==============================
#   FEATURE EXTRACTION & UTILS
# ==============================
def get_hsv_mean(rgb_image, mask):
    hsv_image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2HSV)
    region = hsv_image[mask > 0]
    if len(region) == 0:
        return (0, 0, 0)
    mean_h = np.mean(region[:, 0])
    mean_s = np.mean(region[:, 1])
    mean_v = np.mean(region[:, 2])
    return (mean_h, mean_s, mean_v)

def get_contour_features(mask):
    mask_uint8 = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return 0, 0, 1
    max_contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(max_contour)
    perimeter = cv2.arcLength(max_contour, True)
    x, y, w, h = cv2.boundingRect(max_contour)
    aspect_ratio = max(w, h) / max(1, min(w, h))
    return area, perimeter, aspect_ratio

# ==============================
#   CLASS HEURISTICS & CLASSIFICATION
# ==============================
def is_road_mask(rgb_image, mask):
    h, s, v = get_hsv_mean(rgb_image, mask)
    area, _, ar = get_contour_features(mask)
    if s < 50 and 70 <= v <= 210 and 1000 < area < 1e6:
        return True
    return False

def is_green_area_mask(rgb_image, mask):
    h, s, v = get_hsv_mean(rgb_image, mask)
    area, _, ar = get_contour_features(mask)

    hue_condition = (20 <= h <= 100)
    sat_condition = (s > 10)
    val_condition = (40 < v < 230)
    area_condition = (area > 300)

    return hue_condition and sat_condition and val_condition and area_condition

def is_building_mask(rgb_image, mask):
    area, perimeter, ar = get_contour_features(mask)
    h, s, v = get_hsv_mean(rgb_image, mask)

    if not (area > 5000 and ar < 2.5):
        return False

    if 20 <= h <= 110:
        return False

    return True
def classify_object(rgb_image, mask):
    """ Classifies objects based on HSV and contour properties. """
    h, s, v = get_hsv_mean(rgb_image, mask)
    area, _, ar = get_contour_features(mask)

    if s < 50 and 70 <= v <= 210 and 1000 < area < 1e6:
        return "road"
    elif (20 <= h <= 100) and (s > 10) and (40 < v < 230) and (area > 300):
        return "green_area"
    elif area > 5000 and ar < 2.5 and not (20 <= h <= 110):
        return "building"
    else:
        return "unknown"

def classify_and_colorize(rgb_image, masks):
    """ Applies classification and color mapping to segmented regions. """
    output_image = rgb_image.copy()
    for mask_data in masks:
        seg = mask_data["segmentation"]
        cls = classify_object(rgb_image, seg)
        if cls == "road":
            output_image[seg > 0] = ROAD_COLOR
        elif cls == "green_area":
            output_image[seg > 0] = GREEN_AREA_COLOR
        elif cls == "building":
            output_image[seg > 0] = BUILDING_COLOR
        else:
            output_image[seg > 0] = UNKNOWN_COLOR
    return output_image
    
# ==============================
#    SAVE FINAL CLASSIFIED IMAGE
# ==============================
def save_classified_image(final_classified_image, save_path):
    Image.fromarray(final_classified_image).save(save_path)
    print(f"Saved classified image at: {save_path}")

# ==============================
#   MAIN PROCESSING PIPELINE
# ==============================
def process_images(image_dir, sam_checkpoint, save_dir, overlap_threshold=0.5):
    mask_generator = load_sam_model(sam_checkpoint)

    os.makedirs(save_dir, exist_ok=True)

    for file_name in os.listdir(image_dir):
        image_path = os.path.join(image_dir, file_name)
        if not file_name.lower().endswith((".png", ".jpg", ".jpeg", ".tif")):
            continue

        try:
            original_image_pil = Image.open(image_path).convert("RGB")
            original_image_np = np.array(original_image_pil)

            resized_image_np = resize_image(original_image_pil)
            masks = generate_masks(resized_image_np, mask_generator)
            segmented_overlay = overlay_masks_on_image(resized_image_np, masks)
            final_classified_image = classify_and_colorize(resized_image_np, masks)

            # Display the images with segmentation
            display_images(original_image_np, resized_image_np, segmented_overlay, final_classified_image)

            # Save the final classified image
            save_classified_image(final_classified_image, os.path.join(save_dir, f"{file_name}_classified.png"))

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

# ==============================
#        EXAMPLE USAGE
# ==============================
if __name__ == "__main__":
    image_dir = "/kaggle/input/testimage-2750/"
    sam_checkpoint = "/kaggle/working/sam_vit_l.pth"
    save_dir = "/kaggle/working/output_sam_1240/"

    process_images(image_dir, sam_checkpoint, save_dir, overlap_threshold=0.5)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

def display_all_images(gt_dir, pred_dir, num_images=23):
    """
    Display ground truth and predicted segmentation masks side by side.

    Args:
    - gt_dir (str): Directory containing ground truth masks.
    - pred_dir (str): Directory containing predicted masks.
    - num_images (int): Number of images to display.
    """
    gt_images = sorted(os.listdir(gt_dir))
    pred_images = sorted(os.listdir(pred_dir))

    # Ensure both directories have images
    if len(gt_images) == 0 or len(pred_images) == 0:
        print("❌ ERROR: One of the directories is empty!")
        return

    # Limit to 23 images or the number available
    num_images = min(num_images, len(gt_images), len(pred_images))

    plt.figure(figsize=(15, num_images * 3))

    for i in range(num_images):
        gt_image_path = os.path.join(gt_dir, gt_images[i])
        pred_image_path = os.path.join(pred_dir, pred_images[i])

        # Read images
        gt_image = cv2.imread(gt_image_path)
        pred_image = cv2.imread(pred_image_path)

        if gt_image is None or pred_image is None:
            print(f"Skipping {gt_images[i]}: Could not load image")
            continue

        gt_image = cv2.cvtColor(gt_image, cv2.COLOR_BGR2RGB)
        pred_image = cv2.cvtColor(pred_image, cv2.COLOR_BGR2RGB)

        # Plot images
        plt.subplot(num_images, 2, 2 * i + 1)
        plt.imshow(gt_image)
        plt.title(f"Ground Truth: {gt_images[i]}")
        plt.axis("off")

        plt.subplot(num_images, 2, 2 * i + 2)
        plt.imshow(pred_image)
        plt.title(f"Predicted: {pred_images[i]}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Call the function with your directories
gt_dir = "/kaggle/input/groundtruth-img-1024/"
pred_dir = "/kaggle/working/output_sam_1240/"

display_all_images(gt_dir, pred_dir, num_images=23)

In [ ]:
import os
import numpy as np
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score

# ==============================
#    STEP 1: PARSE LABEL FILE
# ==============================
def parse_label_file(label_file_path):
    """
    Reads the label file and maps class names to RGB colors and class IDs.
    Returns:
        - CLASS_MAPPING: {class_id: class_name}
        - COLOR_TO_CLASS: {(R,G,B): class_id}
    """
    CLASS_MAPPING = {}
    COLOR_TO_CLASS = {}

    with open(label_file_path, "r") as f:
        lines = f.readlines()

    class_id = 0  # Start numbering classes
    for line in lines:
        if line.startswith("#") or line.strip() == "":
            continue  # Skip comments and empty lines

        parts = line.strip().split(":")
        class_name = parts[0]  # e.g., "road"
        rgb_values = tuple(map(int, parts[1].split(",")))  # e.g., (128,64,128)

        CLASS_MAPPING[class_id] = class_name
        COLOR_TO_CLASS[rgb_values] = class_id
        class_id += 1  # Increment class ID

    return CLASS_MAPPING, COLOR_TO_CLASS

# ==============================
#    STEP 2: CONVERT MASKS TO CLASS IDS
# ==============================
def convert_color_mask_to_class_ids(mask_path, color_map):
    """
    Converts an RGB segmentation mask to a grayscale class ID mask.
    
    Args:
        mask_path (str): Path to the mask image.
        color_map (dict): Mapping from (R,G,B) tuples to class IDs.

    Returns:
        class_mask (numpy array): Grayscale mask where each pixel is a class ID.
    """
    mask = cv2.imread(mask_path)

    # Check if the image was loaded correctly
    if mask is None:
        raise FileNotFoundError(f"❌ ERROR: Could not read {mask_path}. Check the file path.")

    # Convert BGR to RGB
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    height, width, _ = mask_rgb.shape
    class_mask = np.zeros((height, width), dtype=np.uint8)

    for rgb_color, class_id in color_map.items():
        mask_pixels = np.all(mask_rgb == np.array(rgb_color), axis=-1)
        class_mask[mask_pixels] = class_id  # Assign class ID

    return class_mask

# ==============================
#    STEP 3: COMPUTE METRICS
# ==============================
def compute_metrics(gt_mask, pred_mask, num_classes, CLASS_MAPPING):
    """
    Compute IoU, Dice Score, Pixel Accuracy, Precision, Recall, and F1 Score.
    """
    iou_per_class = {}
    dice_per_class = {}
    precision_per_class = {}
    recall_per_class = {}
    f1_per_class = {}

    # Compute IoU and Dice for each class
    for class_id in range(num_classes):
        gt = (gt_mask == class_id)
        pred = (pred_mask == class_id)

        intersection = np.logical_and(gt, pred).sum()
        union = np.logical_or(gt, pred).sum()

        iou_per_class[CLASS_MAPPING[class_id]] = intersection / union if union > 0 else 0.0
        dice_per_class[CLASS_MAPPING[class_id]] = (2 * intersection) / (gt.sum() + pred.sum() + 1e-6) if (gt.sum() + pred.sum()) > 0 else 0.0

        # Compute Precision, Recall, and F1 for each class
        precision_per_class[CLASS_MAPPING[class_id]] = precision_score(gt.flatten(), pred.flatten(), average='binary', zero_division=0)
        recall_per_class[CLASS_MAPPING[class_id]] = recall_score(gt.flatten(), pred.flatten(), average='binary', zero_division=0)
        f1_per_class[CLASS_MAPPING[class_id]] = f1_score(gt.flatten(), pred.flatten(), average='binary', zero_division=0)

    # Compute Pixel Accuracy
    pixel_accuracy = (gt_mask == pred_mask).sum() / gt_mask.size

    return iou_per_class, dice_per_class, pixel_accuracy, precision_per_class, recall_per_class, f1_per_class

# ==============================
#    STEP 4: EVALUATE SEGMENTATION
# ==============================
def evaluate_segmentation(gt_dir, pred_dir, label_file_path):
    """
    Evaluate segmentation results for all images in a directory.
    """
    CLASS_MAPPING, COLOR_TO_CLASS = parse_label_file(label_file_path)

    # Debugging: Print directories
    print("\n=== DEBUG: Listing Files ===")
    print("Ground Truth Directory:", gt_dir)
    print("Predicted Directory:", pred_dir)

    # Check if directories exist and contain images
    if not os.listdir(gt_dir):
        print("❌ ERROR: Ground truth directory is empty.")
        return

    if not os.listdir(pred_dir):
        print("❌ ERROR: Predicted directory is empty.")
        return

    iou_scores = []
    dice_scores = []
    pixel_accuracies = []
    precision_scores = []
    recall_scores = []
    f1_scores = []

    image_names = os.listdir(gt_dir)

    for image_name in image_names:
        gt_mask_path = os.path.join(gt_dir, image_name)
        pred_mask_path = os.path.join(pred_dir, image_name.replace(".png", ".png_classified.png"))  # Adjust if needed

        print(f"\nProcessing: {image_name}")
        print(f"GT Path: {gt_mask_path}")
        print(f"Pred Path: {pred_mask_path}")

        # Ensure files exist
        if not os.path.exists(gt_mask_path):
            print(f"❌ ERROR: Ground truth mask not found for {image_name}. Skipping.")
            continue

        if not os.path.exists(pred_mask_path):
            print(f"❌ ERROR: Predicted mask not found for {image_name}. Skipping.")
            continue

        try:
            # Convert masks to class IDs
            gt_class_mask = convert_color_mask_to_class_ids(gt_mask_path, COLOR_TO_CLASS)
            pred_class_mask = convert_color_mask_to_class_ids(pred_mask_path, COLOR_TO_CLASS)

            print(f"✅ Successfully loaded masks for {image_name}")
            print(f"GT Mask Shape: {gt_class_mask.shape}, Pred Mask Shape: {pred_class_mask.shape}")

            # Compute metrics
            iou, dice, accuracy, precision, recall, f1 = compute_metrics(gt_class_mask, pred_class_mask, len(CLASS_MAPPING), CLASS_MAPPING)

            # Store results
            iou_scores.append(iou)
            dice_scores.append(dice)
            pixel_accuracies.append(accuracy)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)

            print(f"IoU: {iou}")
            print(f"Dice Score: {dice}")
            print(f"Pixel Accuracy: {accuracy:.4f}")
            print(f"Precision, Recall, F1: {precision}, {recall}, {f1}")

        except Exception as e:
            print(f"❌ ERROR processing {image_name}: {e}")

    # Compute Mean Metrics
    if iou_scores:
        mean_iou = {k: np.mean([img[k] for img in iou_scores]) for k in iou_scores[0]}
        mean_dice = {k: np.mean([img[k] for img in dice_scores]) for k in dice_scores[0]}
        mean_pixel_acc = np.mean(pixel_accuracies)

        # Compute Mean Precision, Recall, and F1 for each class
        mean_precision = {class_name: np.mean([prf[class_name] for prf in precision_scores]) for class_name in CLASS_MAPPING.values()}
        mean_recall = {class_name: np.mean([prf[class_name] for prf in recall_scores]) for class_name in CLASS_MAPPING.values()}
        mean_f1 = {class_name: np.mean([prf[class_name] for prf in f1_scores]) for class_name in CLASS_MAPPING.values()}

        print("\n=== Overall Evaluation ===")
        print(f"Mean IoU: {mean_iou}")
        print(f"Mean Dice Score: {mean_dice}")
        print(f"Mean Pixel Accuracy: {mean_pixel_acc:.4f}")
        print(f"Mean Precision: {mean_precision}")
        print(f"Mean Recall: {mean_recall}")
        print(f"Mean F1 Score: {mean_f1}")
    else:
        print("⚠ No valid images processed. Check your file paths and formats.")

# ==============================
#    STEP 5: RUN EVALUATION
# ==============================
evaluate_segmentation(
    gt_dir="/kaggle/input/groundtruth-img-1024/",
    pred_dir="/kaggle/working/output_sam_1240/",
    label_file_path="/kaggle/input/labels/labelmap.txt"
)

In [ ]:
import matplotlib.font_manager as fm

# Provide the path to your LinLibertine_RBah.ttf file
font_path = "/kaggle/input/font-style/LinLibertine_RBah.ttf"  # update this path

# Add the font to the font manager
fm.fontManager.addfont(font_path)

# Retrieve the font's name from the file
libertine_name = fm.FontProperties(fname=font_path).get_name()
print("Font name detected:", libertine_name)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import matplotlib.font_manager as fm

# Set Matplotlib to use the detected font and desired font size
plt.rcParams["font.family"] = "Linux Libertine"
plt.rcParams["font.size"] = 16 

def resize_image(image, target_size=(1024, 1024)):
    """
    Resize a PIL image to the target size and return as a NumPy array.
    """
    return np.array(image.resize(target_size, Image.LANCZOS))

def load_image_with_placeholder(image_path, placeholder_text="Missing", target_size=(1024,1024)):
    """
    Attempts to load an image using OpenCV. If the image is not found,
    returns a white placeholder image with the provided text.
    """
    if os.path.exists(image_path):
        img = cv2.imread(image_path)
        if img is not None:
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Create a white placeholder image and add text
    placeholder = np.full((target_size[1], target_size[0], 3), 255, dtype=np.uint8)
    cv2.putText(placeholder, placeholder_text, (50, target_size[1] // 2),
                cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 3, cv2.LINE_AA)
    return placeholder

def display_and_save_single_image(orig_dir, gt_dir, pred_dir, filename, output_path):
    """
    Displays and saves a composite image for a single sample.
    The composite shows:
      - Resized Image (obtained by resizing the original image)
      - Ground Truth mask
      - SAM+Heuristics predicted output (or a placeholder if missing)
    with bold titles above each image.
    
    Args:
      orig_dir (str): Directory containing the original images.
      gt_dir (str): Directory containing the ground truth masks.
      pred_dir (str): Directory containing the SAM+Heuristics predicted masks.
      filename (str): Filename of the image to process (e.g., "img_31.png").
      output_path (str): Full output path (including filename) to save the composite image.
    """
    # Build full paths for the original and ground truth images
    original_path = os.path.join(orig_dir, filename)
    gt_path = os.path.join(gt_dir, filename)
    
    # For the predicted file, assume its name is the original filename appended with "_classified.png"
    pred_filename = f"{filename}_classified.png"
    pred_path = os.path.join(pred_dir, pred_filename)
    
    # Ensure the original and ground truth files exist
    if not os.path.exists(original_path):
        raise FileNotFoundError(f"Original file not found: {original_path}")
    if not os.path.exists(gt_path):
        raise FileNotFoundError(f"Ground truth file not found: {gt_path}")
    
    # For the predicted file, if missing, a placeholder will be used.
    if not os.path.exists(pred_path):
        print(f"Warning: Predicted file not found: {pred_path}. Using placeholder image.")
    
    # Load and resize the original image using PIL
    orig_img_pil = Image.open(original_path).convert("RGB")
    resized_img = resize_image(orig_img_pil)  # Resized to 1024x1024
    
    # Load ground truth and predicted images (or placeholder) using OpenCV
    gt_img = load_image_with_placeholder(gt_path, placeholder_text="GT Missing")
    pred_img = load_image_with_placeholder(pred_path, placeholder_text="Prediction Missing")
    
    # Create a figure with one row and three columns
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    titles = ["Resized Image 1024 x 1024", "Groundtruth Image", "SAM+Heuristics Output"]
    images = [resized_img, gt_img, pred_img]
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=16, fontweight="bold", pad=20)
        ax.axis("off")
    
    plt.tight_layout()
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    # Save the composite image in PDF format with high resolution (300 DPI)
    plt.savefig(output_path, dpi=300, bbox_inches="tight", format="pdf")
    plt.show()

# -------------------------------
# Example usage:
# -------------------------------
# Directories for the images:
orig_dir = "/kaggle/input/testimage-2750/"           # Original images directory
gt_dir = "/kaggle/input/groundtruth-img-1024/"         # Ground truth masks directory
pred_dir = "/kaggle/working/output_sam_1240/"          # Predicted SAM+Heuristics outputs directory

# Specify the image filename to process
filename = "img_31.png"  # Replace with your desired image filename

# Define the output path for the composite image.
# Ensure the output file has a .pdf extension
output_composite_path = os.path.join("/kaggle/working/composite_outputs", f"composite_{os.path.splitext(filename)[0]}.pdf")

# Create, display, and save the composite image
display_and_save_single_image(orig_dir, gt_dir, pred_dir, filename, output_composite_path)
print(f"Saved composite image at: {output_composite_path}")


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import matplotlib.font_manager as fm

# Set Matplotlib to use the detected font and desired font size
plt.rcParams["font.family"] = "Linux Libertine"
plt.rcParams["font.size"] = 16 

def resize_image(image, target_size=(1024, 1024)):
    """
    Resize a PIL image to the target size and return as a NumPy array.
    """
    return np.array(image.resize(target_size, Image.LANCZOS))

def load_image_with_placeholder(image_path, placeholder_text="Missing", target_size=(1024,1024)):
    """
    Attempts to load an image using OpenCV. If the image is not found,
    returns a white placeholder image with the provided text.
    """
    if os.path.exists(image_path):
        img = cv2.imread(image_path)
        if img is not None:
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Create a white placeholder image and add text
    placeholder = np.full((target_size[1], target_size[0], 3), 255, dtype=np.uint8)
    cv2.putText(placeholder, placeholder_text, (50, target_size[1] // 2),
                cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 3, cv2.LINE_AA)
    return placeholder

def display_and_save_single_image(orig_dir, gt_dir, pred_dir, filename, output_path):
    """
    Displays and saves a composite image for a single sample.
    The composite shows:
      - Resized Image (obtained by resizing the original image)
      - Ground Truth mask
      - SAM+Heuristics predicted output (or a placeholder if missing)
    with bold titles above each image.
    
    Args:
      orig_dir (str): Directory containing the original images.
      gt_dir (str): Directory containing the ground truth masks.
      pred_dir (str): Directory containing the SAM+Heuristics predicted masks.
      filename (str): Filename of the image to process (e.g., "img_31.png").
      output_path (str): Full output path (including filename) to save the composite image.
    """
    # Build full paths for the original and ground truth images
    original_path = os.path.join(orig_dir, filename)
    gt_path = os.path.join(gt_dir, filename)
    
    # For the predicted file, assume its name is the original filename appended with "_classified.png"
    pred_filename = f"{filename}_classified.png"
    pred_path = os.path.join(pred_dir, pred_filename)
    
    # Ensure the original and ground truth files exist
    if not os.path.exists(original_path):
        raise FileNotFoundError(f"Original file not found: {original_path}")
    if not os.path.exists(gt_path):
        raise FileNotFoundError(f"Ground truth file not found: {gt_path}")
    
    # For the predicted file, if missing, a placeholder will be used.
    if not os.path.exists(pred_path):
        print(f"Warning: Predicted file not found: {pred_path}. Using placeholder image.")
    
    # Load and resize the original image using PIL
    orig_img_pil = Image.open(original_path).convert("RGB")
    resized_img = resize_image(orig_img_pil)  # Resized to 1024x1024
    
    # Load ground truth and predicted images (or placeholder) using OpenCV
    gt_img = load_image_with_placeholder(gt_path, placeholder_text="GT Missing")
    pred_img = load_image_with_placeholder(pred_path, placeholder_text="Prediction Missing")
    
    # Create a figure with one row and three columns
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    titles = ["Resized Image 1024 x 1024", "Groundtruth Image", "SAM+Heuristics Output"]
    images = [resized_img, gt_img, pred_img]
    
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=16, fontweight="bold", pad=20)
        ax.axis("off")
    
    plt.tight_layout()
    
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    # Save the composite image in PDF format with high resolution (300 DPI)
    plt.savefig(output_path, dpi=300, bbox_inches="tight", format="pdf")
    plt.show()

# -------------------------------
# Example usage:
# -------------------------------
# Directories for the images:
orig_dir = "/kaggle/input/testimage-2750/"           # Original images directory
gt_dir = "/kaggle/input/groundtruth-img-1024/"         # Ground truth masks directory
pred_dir = "/kaggle/working/output_sam_1240/"          # Predicted SAM+Heuristics outputs directory

# Specify the image filename to process
filename = "img_37.png"  # Replace with your desired image filename

# Define the output path for the composite image.
# Ensure the output file has a .pdf extension
output_composite_path = os.path.join("/kaggle/working/composite_outputs", f"composite_{os.path.splitext(filename)[0]}.pdf")

# Create, display, and save the composite image
display_and_save_single_image(orig_dir, gt_dir, pred_dir, filename, output_composite_path)
print(f"Saved composite image at: {output_composite_path}")
